# More Pandas



In [ ]:
import pandas as pd

In [ ]:
# Importing datasets
pop = pd.read_csv("pop.csv")

## Recap: Data Wrangling

I got the population data from [here](https://data.gov.tw/dataset/67531). But once again it is not in the format I like. 

Some rules of thumb that I go by:
* One column represents one variable
* One observation, one row
* One cell, one value
* Do not duplicate things unless this is absolutely necessary

Currently, "男女合計" is the sum of "男" and "女" and "新竹市" is the sum of the three regions, and I don't like it.

In [ ]:
pop.head()

In [ ]:
# Checking dtypes
pop.dtypes

In [ ]:
# Are there missing variables?
pop.isna().sum()

In [ ]:
# I want to keep only "男女合計"
# This will ask python to: 
# 1. Take the "性別" column; 
# 2. if the value equals to "男女合計", return True; 
# 3. use the results to select the rows from the dataset; 
# 4. put that back as "pop"

pop = pop[pop['性別'] == '男女合計']

In [ ]:
# Currently the spreadsheet has duplicated data, '新竹市' is a total of three regions: 東區, 北區, and 香山
# I want to toss that away (why would we need duplicated data anyways?)

pop = pop[pop['區域別'] != '新竹市']

In [ ]:
# This is much more compact now

pop.head()

## Split-Apply-Combine

### Step 1: Group

Technically you don't need to do this step separately

In [ ]:
pop.groupby('區域別')

### Step 2: Apply a function

In [ ]:
# Group by region, take population, apply mean
# Note Pandas will do Step 3 (Combine) automatically
pop.groupby('區域別')['人口數'].mean()

In [ ]:
# You store the output somewhere
results = pop.groupby('年月')['遷入人數'].mean()

In [ ]:
# This is a Pandas series, it behaves like a list
type(results)

In [ ]:
# We can apply what we learnt last week to sort them
results.sort_values(ascending=False)

In [ ]:
# More data wrangling, let's split it into year and month
pop['year'] = pop['年月'].astype(str).str[:3]
pop['month'] = pop['年月'].astype(str).str[3:]

`pop['年月'].astype(str).str[:3]` basically means:
1. Take the column `pop['年月']`
2. Change the values as string `.astype(str)`
3. Apply a string function (taking the first 3 characters) `.str[:3]`


In [ ]:
# When did most people move to Hsinchu?
pop.groupby('year')['遷入人數'].mean()

In [ ]:
# Is it seasonal?
pop.groupby('month')['遷入人數'].mean()

In [ ]:
# For a fuller desciprtion by group
pop.groupby('year')['遷入人數'].describe()

In [ ]:
pop

## Merging

Often, data is stored across multiple tables. Merging allows us to combine them based on common columns.

Let's import another dataset:

In [ ]:
food = pd.read_csv("food.csv")
food.head()

In [ ]:
# Here's how I can count how many recommended stores per region

food.groupby('行政區').size()

In [ ]:
# By default, pandas will put your grouping column into the index, not as a normal column.
# `reset_index()` moves the index back into a column.

food_region = food.groupby('行政區').size().reset_index(name='food_count')
print(type(food_region))
food_region

In [ ]:
# Wait, there is an error, one is noted as 香山區, another 香山.
food_region['行政區'] = food_region['行政區'].replace({'香山區': '香山'})
food_region

In [ ]:
#Merging on different keys - left
pop_with_food = pd.merge(pop, food_region, how = 'left', left_on = '區域別', right_on='行政區')
len(pop_with_food)

In [ ]:
pop_with_food.head()

In [ ]:
# Let's find out how many recommended food per person?

pop_with_food['food_index'] = pop_with_food['food_count'] / pop_with_food['人口數']

In [ ]:
pop_with_food.sort_values(by = 'food_index', ascending=False) # Now sort the dataframe by the index that we created

### Basic Plotting

When plotting Chinese characters, it is very likely that you would come across some errors.

To fix this, you need to add the fonts:

In [ ]:
# Just run this, it will search for one of the fonts that you have installed in your system
# PS: doing things in non-English is always a pain

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def setup_cjk_font():
    candidates = [
        'Noto Sans CJK TC',
        'Noto Sans CJK SC',
        'Noto Sans CJK JP',
        'Microsoft JhengHei',
        'PingFang TC',
        'SimHei',
        'Arial Unicode MS'
    ]
    
    available = {f.name for f in fm.fontManager.ttflist}
    
    for font in candidates:
        if font in available:
            plt.rcParams['font.family'] = 'sans-serif'
            plt.rcParams['font.sans-serif'] = [font]
            plt.rcParams['axes.unicode_minus'] = False
            print(f'Using font: {font}')
            return font
    
    print('No CJK font found by matplotlib.')
    return None

setup_cjk_font()

Now back to the real plotting, generating plots in Pandas is extremely easy, you just need to add `.plot()`.

In [ ]:
food.groupby('行政區').size().plot(kind='bar')

In [ ]:
pop['遷入人數'].hist()

In [ ]:
pop.boxplot(column='遷入人數', by='區域別')

In [ ]:
pop.plot(kind='scatter', x='遷出人數', y='遷入人數')

In [ ]:
# Sometimes it is a good idea to create a dataframe before plotting
df_plot = pop.groupby(['year'])['遷入人數'].sum().reset_index()

In [ ]:
df_plot.plot(x='year', y='遷入人數', kind='line')

## Some Last Words

Pandas is very convenient for quick plots, just to get to know your data. But the functions can be limiting and it is, in my opinion, quite ugly.

Next week we will learn about how to do plots for real using seaborn!